In [ ]:
import heapq


def heuristic(pos, goals):
    return min(abs(pos[0]-g[0]) + abs(pos[1]-g[1]) for g in goals)

def multi_goal_best_first(maze, start, goals):
    rows, cols = len(maze), len(maze[0])
    directions = [(-1,0),(1,0),(0,-1),(0,1)]
    
    pq = []
    heapq.heappush(pq, (0, start, tuple(goals), [start]))
    
    visited = set()
    
    while pq:
        _, current, remaining_goals, path = heapq.heappop(pq)
        
        if not remaining_goals:
            return path
        
        state = (current, remaining_goals)
        if state in visited:
            continue
        visited.add(state)
        
        x, y = current
        
        for dx, dy in directions:
            nx, ny = x+dx, y+dy
            
            if 0 <= nx < rows and 0 <= ny < cols and maze[nx][ny] != 1:
                new_goals = tuple(g for g in remaining_goals if g != (nx, ny))
                h = heuristic((nx, ny), new_goals) if new_goals else 0
                heapq.heappush(pq, (h, (nx, ny), new_goals, path + [(nx, ny)]))
    
    return None



maze = [
    [0,0,0,0],
    [1,1,0,1],
    [0,0,0,0],
    [0,1,1,0]
]

start = (0,0)
goals = [(2,3), (3,3)]

path = multi_goal_best_first(maze, start, goals)
print("Mult-goal Path:", path)

Mult-goal Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (3, 3)]


In [ ]:
import heapq
import math

def distance(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

def greedy_delivery(start, deliveries):
    current = start
    current_time = 0
    route = [start]
    
    deliveries = deliveries.copy()
    
    while deliveries:
        pq = []
        
        for point, window in deliveries:
            travel_time = distance(current, point)
            arrival_time = current_time + travel_time
            
            if arrival_time <= window[1]: 
                priority = (window[1] - arrival_time) 
                heapq.heappush(pq, (priority, travel_time, point, window))
        
        if not pq:
            print("No feasible delivery!")
            return route
        
        _, travel_time, point, window = heapq.heappop(pq)
        
        current_time += travel_time
        current = point
        route.append(point)
        deliveries.remove((point, window))
    
    return route



start = (0,0)
deliveries = [
    ((2,3),(0,10)),
    ((5,2),(0,8)),
    ((1,4),(0,6))
]

print("Optimzed Delivry Route:", greedy_delivery(start, deliveries))

In [2]:
from typing import Dict, List
from queue import PriorityQueue



deliveries: Dict[str, int] = {
    'A': 10,
    'B': 5,
    'C': 8,
    'D': 3,
    'E': 7
}


graph: Dict[str, Dict[str, int]] = {
    'Start': {'A': 2, 'B': 4},
    'A': {'C': 3},
    'B': {'C': 2, 'D': 5},
    'C': {'E': 4},
    'D': {'E': 1},
    'E': {}
}


def greedy_delivery(graph: Dict[str, Dict[str, int]],
                    deliveries: Dict[str, int],
                    start: str) -> List[str]:


    frontier = PriorityQueue()

   
    for location, deadline in deliveries.items():
        frontier.put((deadline, location))

    visited = set()
    current_location = start
    route = [start]

    current_time = 0

    while not frontier.empty():

        deadline, location = frontier.get()

        if location in visited:
            continue


        if location in graph[current_location]:
            travel_time = graph[current_location][location]
        else:
            travel_time = 1 

        current_time += travel_time

       
        if current_time <= deadline:
            route.append(location)
            visited.add(location)
            current_location = location
        else:
            print(f"Skipped {location} (missed deadline)")

    return route



route = greedy_delivery(graph, deliveries, 'Start')
print("Delivery Route:", route)

Delivery Route: ['Start', 'D', 'B', 'E', 'C', 'A']


In [ ]:
import heapq
import random
import time
import threading
import math

# -----------------------------
# Graph Definition
# -----------------------------

graph = {
    'A': {'B': 4, 'C': 2},
    'B': {'A': 4, 'D': 5, 'E': 10},
    'C': {'A': 2, 'D': 8},
    'D': {'B': 5, 'C': 8, 'E': 2, 'F': 6},
    'E': {'B': 10, 'D': 2, 'F': 3},
    'F': {'D': 6, 'E': 3}
}

# Node positions for heuristic
positions = {
    'A': (0, 0),
    'B': (2, 3),
    'C': (2, 0),
    'D': (4, 2),
    'E': (6, 3),
    'F': (7, 0)
}

start = 'A'
goal = 'F'

# -----------------------------
# Heuristic Function
# -----------------------------

def heuristic(a, b):
    x1, y1 = positions[a]
    x2, y2 = positions[b]
    return math.sqrt((x1-x2)**2 + (y1-y2)**2)

# -----------------------------
# A* Search
# -----------------------------

def astar(graph, start, goal):

    open_set = []
    heapq.heappush(open_set, (0, start))

    came_from = {}

    g_cost = {node: float('inf') for node in graph}
    g_cost[start] = 0

    f_cost = {node: float('inf') for node in graph}
    f_cost[start] = heuristic(start, goal)

    while open_set:

        current = heapq.heappop(open_set)[1]

        if current == goal:
            return reconstruct_path(came_from, current)

        for neighbor in graph[current]:

            tentative_g = g_cost[current] + graph[current][neighbor]

            if tentative_g < g_cost[neighbor]:

                came_from[neighbor] = current
                g_cost[neighbor] = tentative_g
                f_cost[neighbor] = tentative_g + heuristic(neighbor, goal)

                heapq.heappush(open_set, (f_cost[neighbor], neighbor))

    return None

# -----------------------------
# Reconstruct Path
# -----------------------------

def reconstruct_path(came_from, current):

    path = [current]

    while current in came_from:
        current = came_from[current]
        path.append(current)

    path.reverse()
    return path

# -----------------------------
# Random Edge Updates
# -----------------------------

def update_edge_costs():

    while True:

        nodes = list(graph.keys())

        u = random.choice(nodes)
        v = random.choice(list(graph[u].keys()))

        new_cost = random.randint(1, 15)

        graph[u][v] = new_cost
        graph[v][u] = new_cost

        print(f"\nEdge cost updated: {u}-{v} = {new_cost}")

        time.sleep(4)

# -----------------------------
# Start Dynamic Updates
# -----------------------------

t = threading.Thread(target=update_edge_costs)
t.daemon = True
t.start()

# -----------------------------
# Continuous Path Planning
# -----------------------------

while True:

    path = astar(graph, start, goal)

    print("\nCurrent Best Path:", path)

    time.sleep(3)

In [1]:
from queue import PriorityQueue

graph = {
    'A': {'B': 2, 'C': 1},
    'B': {'D': 4, 'E': 3},
    'C': {'F': 1, 'G': 5},
    'D': {'H': 2},
    'E': {},
    'F': {'I': 6},
    'G': {},
    'H': {},
    'I': {}
}

goals = {'E','H','I'}   # multiple goals


class Node:

    def __init__(self, position, parent=None, goals_visited=None):
        self.position = position
        self.parent = parent
        self.path_cost = 0
        self.goals_visited = goals_visited or set()

    def __lt__(self, other):
        return self.path_cost < other.path_cost


def multi_goal_search(start, goals, graph):

    frontier = PriorityQueue()

    start_node = Node(start, None, set())
    frontier.put(start_node)

    visited_states = set()

    while not frontier.empty():

        current = frontier.get()

        state = (current.position, tuple(sorted(current.goals_visited)))

        if state in visited_states:
            continue

        visited_states.add(state)

        # check goal
        if current.position in goals:
            current.goals_visited = set(current.goals_visited)
            current.goals_visited.add(current.position)

        # if all goals reached
        if current.goals_visited == goals:

            path = []
            cost = current.path_cost

            while current:
                path.append(current.position)
                current = current.parent

            return cost, path[::-1]

        # expand neighbors
        for neighbor, weight in graph[current.position].items():

            child_goals = set(current.goals_visited)

            child = Node(neighbor, current, child_goals)
            child.path_cost = current.path_cost + weight

            frontier.put(child)

    return None


result = multi_goal_search('A', goals, graph)

print("Result:", result)

Result: None
